# Chapter 6
Hypothesis 3 (field data):
The time-lapse phase-plane approach generalises beyond idealised synthetic
models to real borehole GPR field data, where neither the target geometry
nor the displacement is externally controlled.

_____
# Chapter 6.0 Setting up the Python Notebook
- load in functions
- pre-process the raw field profiles
- load in the pre-computed migration / back-propagation results

In [ ]:
import os, sys
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')   # avoid libomp double-init crash (pylops + MKL)
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.patches import Rectangle, Patch
from scipy.signal.windows import tukey
import pyvista

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from gdp.data_io import load_mala
from gdp.preprocessing.filtering import filter_data, remove_mean
from gdp.preprocessing.gain import linear_gain
from gdp.preprocessing.image_processing import remove_svd
from gdp.preprocessing.trace_ops import align_traces

from IPython.display import display
from helper_functions.figures import setup_autosave

# Every plt.show() below is auto-saved to TimeLapse_Figures/Hypothesis_3/<category>/<NNN>_<title>.png
setup_autosave(study="Hypothesis_3", prefix="H3_")

# ── Dataset paths and acquisition parameters (Ploemeur borehole GPR campaign, 06-06-2016) ──
DATA    = ROOT / 'fielddata' / 'raw_data' / '060616'
OUT_DIR = ROOT / 'fielddata' / 'output'   # Kirchhoff / Gazdag / back-propagation products,
                                           # already computed by FieldData_Playground.ipynb
                                           # (Kirchhoff & Gazdag: pylops, seconds per profile;
                                           # back-propagation: an external gprMax FDTD run per
                                           # profile, hours -- both are simply loaded below, not
                                           # recomputed) -- ready to be imported.

v         = 0.10   # propagation velocity [m/ns]
dL        = 0.05   # trace spacing along the borehole [m]
max_d     = 85.0   # deepest receiver position [m]
rad_cut   = 300    # number of time samples kept (migrated-image radial axis length)
f0_mig    = 0.10   # migration centre frequency [GHz]
sc        = 1.2    # display amplitude scaling (x max|amplitude|)

REF_RUN  = 0                        # pre-injection reference profile
PROFILES = [1, 3, 8, 20, 38]        # begin/end markers of every experimental phase
STAGE_PAIRS = [
    ('Pushing', 1, 3),
    ('Chasing', 3, 8),
    ('Waiting', 8, 20),
    ('Pulling', 20, 38),
]
STAGE_COLOURS = {'Pushing': '#aed6f1', 'Chasing': '#a9dfbf',
                  'Waiting': '#f9e79f', 'Pulling': '#f1948a'}

# Fixed ROI used for every stage: the coherent reflector sits at 70-79 m depth,
# 4.5-7.5 m radial distance in every stage and every migration method (confirmed
# visually in Chapter 6.3 below).
ROI_PHYS = (70.0, 79.0, 4.5, 7.5)   # (depth_min, depth_max, radial_min, radial_max) [m]

kz_c    = 2.0 * np.pi * f0_mig / v          # central wavenumber, Kirchhoff/Gazdag image domain
kz_c_bp = 2.0 * np.pi * f0_mig / (v / 2.0)  # central wavenumber, gprMax back-propagation domain

METHODS = ['Kirchhoff-BP', 'Gazdag', 'Back-propagation']
_METHOD_FILE = {'Kirchhoff-BP': 'kirchhoff_bp', 'Gazdag': 'gazdag'}

_prof_name = lambda n: 'prof' if n == 0 else f'prof{n}'


def phys_to_pix(z_axis, x_axis, z_min, z_max, x_min, x_max):
    """Physical ROI (metres) -> pixel index box (z0, z1, x0, x1), z1/x1 exclusive.
    Works for both increasing and decreasing z_axis (Kirchhoff/Gazdag depth arrays
    run 85 -> 0; gprMax back-propagation depth arrays run 0 -> 86)."""
    rows = np.where((z_axis >= z_min) & (z_axis <= z_max))[0]
    cols = np.where((x_axis >= x_min) & (x_axis <= x_max))[0]
    return int(rows.min()), int(rows.max()) + 1, int(cols.min()), int(cols.max()) + 1


def monogenic_envelope(img):
    """2-D generalisation of the Hilbert envelope: sqrt(f^2 + Rz^2 + Rx^2), where
    (Rz, Rx) is the Riesz transform of img. Phase-invariant -- highlights coherent
    energy regardless of wavelet polarity; used in Chapter 6.3 to confirm the fixed
    ROI above sits on a single coherent patch in every stage/method."""
    nz, nx = img.shape
    KZ, KX = np.meshgrid(np.fft.fftfreq(nz), np.fft.fftfreq(nx), indexing='ij')
    K = np.sqrt(KZ**2 + KX**2); K[0, 0] = 1.0
    F = np.fft.fft2(img)
    Rz = np.real(np.fft.ifft2((-1j * KZ / K) * F))
    Rx = np.real(np.fft.ifft2((-1j * KX / K) * F))
    return np.sqrt(img**2 + Rz**2 + Rx**2)


print('Setup ready.')
print(f'PROFILES = {PROFILES}')
print('STAGE_PAIRS =', [(s[0], s[1], s[2]) for s in STAGE_PAIRS])
print(f'Fixed ROI (depth, radial) [m] = {ROI_PHYS}')

_____
# Chapter 6.1: Pre-processing and Profile Visualisation

11-step field pre-processing chain: bandpass filter (0.02-0.20 GHz), DC/direct-wave
removal, trace alignment (x5 upsampling) to the pre-injection reference profile, SVD
rank-1 direct-wave suppression, and linear spherical-gain correction. Shown below for
the reference profile and the 5 profiles that mark the start/end of every operational
stage (Push, Chase, Wait, Pull).

In [ ]:
ref_raw, _info = load_mala(str(DATA / _prof_name(REF_RUN)), return_object=False)
sf       = _info['frequency (GHz)']
n_traces = ref_raw.shape[1]
samples  = ref_raw.shape[0]

t     = np.arange(1, samples + 1) / sf                       # time axis [ns]
depth = np.linspace(max_d, max_d - n_traces * dL, n_traces)  # receiver depth [m], decreasing 85->0
t_mig = t[:rad_cut]
x_img = np.linspace(0, v * t_mig[-1] / 2, rad_cut)            # migrated-image radial axis [m]

ref_bp              = filter_data(ref_raw, fq=(0.02, 0.2), sfreq=sf, btype='bandpass')
ref_dc, _           = remove_mean(ref_bp, 299, 517)
ref_aligned, _, _   = align_traces(ref_dc, ref_dc, upsample=5, normalize=False, align_reference=True)
ref_svd, _          = remove_svd(ref_aligned, low_s=0, high_s=1)
ref_gain, _         = linear_gain(ref_svd, t)
Dt_ref              = ref_gain[:rad_cut, :].T   # (n_traces, rad_cut)


def preprocess_profile(run):
    """Pre-process one field profile (sec:hyp3-fd-setup): bandpass -> DC/direct-wave
    removal -> trace alignment (to the reference) -> SVD rank-1 removal -> linear
    gain. Returns (Dt, z_bh): the processed B-scan (n, rad_cut) and its receiver-depth
    axis. Run once per profile and cached in PROCESSED below, reused by every later
    chapter -- mirrors Hypothesis_1/2's load_study() pattern."""
    if run == REF_RUN:
        return Dt_ref, depth[:n_traces]
    data, _ = load_mala(str(DATA / _prof_name(run)), return_object=False)
    n = min(data.shape[1], n_traces)
    d_bp = filter_data(data[:, :n], fq=(0.02, 0.2), sfreq=sf, btype='bandpass')
    d_dc, _ = remove_mean(d_bp, 299, 517)
    d_aligned, _, _ = align_traces(d_dc, ref_aligned[:, :n], upsample=5, normalize=True, align_reference=False)
    d_svd, _ = remove_svd(d_aligned, low_s=0, high_s=1)
    d_gain, _ = linear_gain(d_svd, t)
    return d_gain[:rad_cut, :].T, depth[:n]


PROCESSED = {run: preprocess_profile(run) for run in [REF_RUN] + PROFILES}
print('Processed:', {k: v[0].shape for k, v in PROCESSED.items()})

# ── Compiled B-scan grid: reference + the 5 stage-boundary profiles ────────────────
rows = [REF_RUN] + PROFILES
fig, axes = plt.subplots(1, len(rows), figsize=(3.4 * len(rows), 6), sharey=True)
for ax, run in zip(axes, rows):
    Dt, z_bh = PROCESSED[run]
    lim = max(sc * np.max(np.abs(Dt)), 1.0)
    ax.imshow(Dt, aspect='auto', cmap='seismic',
              extent=[x_img[0], x_img[-1], z_bh[-1], z_bh[0]],
              vmin=-lim, vmax=lim, origin='upper')
    ax.invert_yaxis()
    ax.set_title('Reference' if run == REF_RUN else f'Profile {run}')
    ax.set_xlabel('Radial distance (m)')
axes[0].set_ylabel('Depth (m)')
fig.suptitle('Processed B-scans -- reference and stage-boundary profiles', y=1.02)
plt.tight_layout(); plt.show(); plt.close(fig)


_____
# Chapter 6.2: Migration of Pre-processed Profiles

Kirchhoff-BP and Gazdag migrations (pylops) and back-propagation $E_z$ focus frames
(external gprMax time-reversal run) for the 5 stage-boundary profiles, imported from
the pre-computed cache. All three are already time-lapse differenced against the
pre-injection reference profile (Kirchhoff/Gazdag: migrated image minus the reference's
own migrated image; back-propagation: the *difference* B-scan `d_svd - ref_svd` was
back-propagated directly, so the returned field is the differenced field already) --
so what is shown below is each profile's migrated anomaly relative to the pre-injection
baseline, not an absolute migration.

In [ ]:
def load_migrated(method, run):
    """Load a cached migrated .npy (already time-lapse differenced against the
    pre-injection reference profile 0)."""
    p = OUT_DIR / 'migrated' / f'{method}_{run}.npy'
    return np.load(p) if p.exists() else None


# ── Back-propagation focus-frame snapshot timing -- must match the parameters used by
# FieldData_Playground.ipynb's snapshot-saving cell (N_SNAP, SNAP_WIN, offset) so the
# same physical instant in the gprMax run is read here. ─────────────────────────────
N_SNAP          = 30
SNAP_WIN        = 1.0    # ns
BP_FOCUS_OFFSET = 28     # snapshot index offset -> the well-focused instant
BP_NEAR_MASK_M  = 1.0    # hide radial dist < this [m]: source-injection halo, not signal

dt_ns_bp   = 1.0 / sf
T_ns_bp    = rad_cut * dt_ns_bp
t0_ns_bp   = 299.0 / sf
t_focus_ns = T_ns_bp - t0_ns_bp
t_start_ns = max(dt_ns_bp, t_focus_ns - SNAP_WIN)
snap_step  = max(1, int((T_ns_bp - t_start_ns) / (max(1, N_SNAP - 1) * dt_ns_bp)))


def _bp_snap_files(run):
    slug = f'prof_{run}'
    snap_dir = OUT_DIR / 'backprop' / slug / f'backprop_{slug}_snaps'
    if not snap_dir.exists():
        return []
    return sorted(snap_dir.glob('bp_snap*.vti'), key=lambda p: int(p.stem.replace('bp_snap', '')))


def load_backprop_focus(run, offset=BP_FOCUS_OFFSET):
    """Load the Ez focus-frame snapshot for one profile's back-propagation run
    (already time-lapse differenced -- see the markdown note above).
    Returns (ez, depth_axis, radial_axis, dx_m) with depth_axis INCREASING (0 m at the
    surface); returns None if that profile's gprMax run hasn't completed yet."""
    snap_files = _bp_snap_files(run)
    if not snap_files:
        return None
    snap_times_ns = t_start_ns + np.arange(len(snap_files)) * snap_step * dt_ns_bp
    idx = min(int(np.argmin(np.abs(snap_times_ns - t_focus_ns))) + offset, len(snap_files) - 1)
    mesh = pyvista.read(str(snap_files[idx]))
    nx_c = mesh.dimensions[0] - 1
    ny_c = mesh.dimensions[1] - 1
    dx_m = float(mesh.spacing[0])
    e_data = np.array(mesh['E-field'])
    ez = e_data[:, 2].reshape(ny_c, nx_c).T   # rows=depth, cols=radial
    depth_axis  = np.linspace(0, nx_c * dx_m, nx_c)
    radial_axis = np.linspace(0, ny_c * dx_m, ny_c)
    return ez, depth_axis, radial_axis, dx_m


# ── Figure: profile / migration / back-propagation compilation grid ────────────────
FIG1_DEPTH_RANGE = (62, 86)
FIG1_RADIAL_RANGE = (0, float(x_img[-1]))
col_titles = ['Processed B-scan', 'Kirchhoff-BP migration', 'Gazdag migration',
              'Back-propagation ($E_z$ focus frame)']

fig1, axes1 = plt.subplots(len(PROFILES), 4, figsize=(17, 3.6 * len(PROFILES)),
                            sharex=True, sharey=True)

for row, run in enumerate(PROFILES):
    Dt_row, z_bh_row = PROCESSED[run]
    lim = max(sc * np.max(np.abs(Dt_row)), 1.0)
    ax = axes1[row, 0]
    ax.imshow(Dt_row, aspect='auto', cmap='seismic',
              extent=[x_img[0], x_img[-1], z_bh_row[-1], z_bh_row[0]],
              vmin=-lim, vmax=lim, origin='upper')
    ax.invert_yaxis()
    ax.set_ylabel(f'Profile {run}\nDepth (m)')

    for col, method in [(1, 'kirchhoff_bp'), (2, 'gazdag')]:
        img = load_migrated(method, run)
        ax = axes1[row, col]
        if img is None:
            ax.text(0.5, 0.5, 'missing', ha='center', va='center', transform=ax.transAxes)
            continue
        n2 = img.shape[0]; z2 = depth[:n2]
        lim2 = max(sc * np.max(np.abs(img)), 1.0)
        ax.imshow(img, aspect='auto', cmap='seismic',
                  extent=[x_img[0], x_img[-1], z2[-1], z2[0]],
                  vmin=-lim2, vmax=lim2, origin='upper')
        ax.invert_yaxis()

    ax = axes1[row, 3]
    res = load_backprop_focus(run)
    if res is None:
        ax.text(0.5, 0.5, 'missing', ha='center', va='center', transform=ax.transAxes)
    else:
        ez, depth_axis, radial_axis, dx_m = res
        mask_px = max(1, round(BP_NEAR_MASK_M / dx_m))
        ez_disp = ez.copy(); ez_disp[:, :mask_px] = 0.0
        clim = np.percentile(np.abs(ez_disp), 100) if ez_disp.any() else 1.0
        ax.imshow(ez_disp, aspect='auto', cmap='seismic',
                  extent=[0, float(radial_axis[-1]), 0, float(depth_axis[-1])],
                  vmin=-clim, vmax=clim, origin='lower')
        ax.invert_yaxis()

    for col in range(4):
        axes1[row, col].set_ylim(FIG1_DEPTH_RANGE[1], FIG1_DEPTH_RANGE[0])
        axes1[row, col].set_xlim(*FIG1_RADIAL_RANGE)
        axes1[row, col].xaxis.set_major_locator(ticker.MultipleLocator(2))
        axes1[row, col].yaxis.set_major_locator(ticker.MultipleLocator(5))
        if row == len(PROFILES) - 1:
            axes1[row, col].set_xlabel('Radial distance (m)')

for col, title in enumerate(col_titles):
    axes1[0, col].set_title(title)

fig1.suptitle('Processed profiles and their migrated / back-propagated counterparts '
              '(Kirchhoff-BP, Gazdag, Back-propagation)', y=1.005)
plt.tight_layout(); plt.show(); plt.close(fig1)


_____
# Chapter 6.3: Region of Influence on the Time-lapse Profiles

A single fixed rectangular ROI (depth 70-79 m, radial distance 4.5-7.5 m) is used for
every stage and every migration method: the reflector sits at the same location in
every profile (Chapter 6.2), so one predefined window suffices -- no auto-detection is
needed. Each panel below is a genuine *time-lapse-of-time-lapse* difference (e.g.
profile 3 minus profile 1), since the Chapter 6.2 products are themselves already
differenced against the pre-injection reference. The monogenic envelope (phase-invariant
amplitude of the 2-D analytic signal) confirms the ROI captures a single coherent patch
in every case.

In [ ]:
def get_crop(method, run_a, run_b, roi=ROI_PHYS):
    """Load two profiles for one migration method, difference them, and crop to the
    ROI. Shared by Chapter 6.3 (diff + envelope display) and Chapter 6.4 (WLS
    phase-plane fit): returns a dict with the cropped base/monitor images, the full
    (uncropped) difference image, both axes, the ROI's pixel box, the grid spacings
    and central wavenumber needed for the WLS fit, and the imshow origin convention."""
    if method == 'Back-propagation':
        res_a, res_b = load_backprop_focus(run_a), load_backprop_focus(run_b)
        if res_a is None or res_b is None:
            return None
        ez_a, depth_ax, radial_ax, dx_m = res_a
        ez_b, _, _, _ = res_b
        nd = min(ez_a.shape[0], ez_b.shape[0]); nr = min(ez_a.shape[1], ez_b.shape[1])
        ez_a, ez_b = ez_a[:nd, :nr], ez_b[:nd, :nr]
        depth_ax, radial_ax = depth_ax[:nd], radial_ax[:nr]
        z0, z1, x0, x1 = phys_to_pix(depth_ax, radial_ax, *roi)
        full_diff = ez_b - ez_a
        return dict(base=ez_a[z0:z1, x0:x1], mon=ez_b[z0:z1, x0:x1], full_diff=full_diff,
                    z_axis=depth_ax, x_axis=radial_ax, roi_px=(z0, z1, x0, x1),
                    dz_g=dx_m, dx_g=dx_m, kz_cent=kz_c_bp, origin='lower')
    else:
        key = _METHOD_FILE[method]
        img_a, img_b = load_migrated(key, run_a), load_migrated(key, run_b)
        if img_a is None or img_b is None:
            return None
        n = min(img_a.shape[0], img_b.shape[0])
        img_a, img_b = img_a[:n], img_b[:n]
        z_axis = depth[:n]
        z0, z1, x0, x1 = phys_to_pix(z_axis, x_img, *roi)
        full_diff = img_b - img_a
        return dict(base=img_a[z0:z1, x0:x1], mon=img_b[z0:z1, x0:x1], full_diff=full_diff,
                    z_axis=z_axis, x_axis=x_img, roi_px=(z0, z1, x0, x1),
                    dz_g=dL, dx_g=float(x_img[1] - x_img[0]), kz_cent=kz_c, origin='upper')


def plot_stage_roi(stage_name, run_a, run_b, roi=ROI_PHYS):
    """Chapter 6.3 driver: one figure per stage -- 3 rows (Kirchhoff-BP, Gazdag,
    Back-propagation) x 2 columns (difference + ROI, monogenic envelope + ROI)."""
    z_min, z_max, x_min, x_max = roi
    fig, axes = plt.subplots(3, 2, figsize=(12, 12), sharex=True, sharey=True)

    for row, method in enumerate(METHODS):
        crop = get_crop(method, run_a, run_b, roi)
        ax_d, ax_e = axes[row, 0], axes[row, 1]
        if crop is None:
            for ax in (ax_d, ax_e):
                ax.text(0.5, 0.5, 'missing data', ha='center', va='center', transform=ax.transAxes)
            continue
        diff, z_axis, x_axis, origin = crop['full_diff'], crop['z_axis'], crop['x_axis'], crop['origin']
        env = monogenic_envelope(diff)
        extent = ([x_axis[0], x_axis[-1], z_axis[-1], z_axis[0]] if origin == 'upper'
                  else [x_axis[0], x_axis[-1], z_axis[0], z_axis[-1]])

        vmax_d = np.percentile(np.abs(diff), 98)
        vmax_e = np.percentile(env, 98)

        im_d = ax_d.imshow(diff, aspect='auto', cmap='RdBu_r', extent=extent,
                            origin=origin, vmin=-vmax_d, vmax=vmax_d)
        plt.colorbar(im_d, ax=ax_d, label='Delta amplitude [a.u.]', fraction=0.046, pad=0.04)
        ax_d.add_patch(Rectangle((x_min, z_min), x_max - x_min, z_max - z_min,
                                  lw=1.5, edgecolor='yellow', facecolor='none'))

        im_e = ax_e.imshow(env, aspect='auto', cmap='inferno', extent=extent,
                            origin=origin, vmin=0, vmax=vmax_e)
        plt.colorbar(im_e, ax=ax_e, label='Monogenic envelope [a.u.]', fraction=0.046, pad=0.04)
        ax_e.add_patch(Rectangle((x_min, z_min), x_max - x_min, z_max - z_min,
                                  lw=1.5, edgecolor='cyan', facecolor='none'))

        for ax in (ax_d, ax_e):
            ax.invert_yaxis(); ax.set_ylim(86, 62); ax.set_xlim(0, float(x_img[-1]))
            ax.xaxis.set_major_locator(ticker.MultipleLocator(2))
            ax.yaxis.set_major_locator(ticker.MultipleLocator(5))
            ax.grid(True, color='grey', lw=0.3, alpha=0.4)
        ax_d.set_ylabel(f'{method}\nDepth (m)')

    axes[0, 0].set_title(f'Difference: profile {run_b} - profile {run_a}')
    axes[0, 1].set_title('Monogenic envelope')
    for ax in axes[-1]:
        ax.set_xlabel('Radial distance (m)')
    fig.suptitle(f'{stage_name} stage (profiles {run_a} -> {run_b}): difference and monogenic '
                 f'envelope, ROI depth {roi[0]}-{roi[1]} m, radial {roi[2]}-{roi[3]} m', y=1.01)
    plt.tight_layout(); plt.show(); plt.close(fig)


for stage_name, run_a, run_b in STAGE_PAIRS:
    plot_stage_roi(stage_name, run_a, run_b)


_____
# Chapter 6.4: Phase-Plane Fit Workflow

The WLS cross-spectrum phase-plane estimator (@sec:meth-phaseplane) is applied to the
ROI-cropped base/monitor pair of every stage, for all three migration methods. Each
figure below shows the full 5-panel diagnostic (cross-spectrum phase, cross-spectrum
energy with the WLS amplitude-threshold contour, the fitted plane, and the 1-D $k_z$
and $k_x$ slices with their fits) for the four operational stages (rows).
Kirchhoff-BP/Gazdag band factors are inherited from the per-pair calibration in
`FieldData_Playground.ipynb` (same profile pairs); back-propagation uses the same
defaults that notebook used for its own diagnostic figures.

In [ ]:
KZ_FAC_OVERRIDE = {'Waiting': 0.35, 'Pulling': 0.45}   # image-domain only; default 0.5
KX_FAC_IMG, WLS_THR_IMG, WLS_POW_IMG = 2.0, 0.20, 3
KX_FAC_BP,  WLS_THR_BP,  WLS_POW_BP  = 1.5, 0.10, 1
PAD_FAC = 10

PHASE_RESULTS = {method: {} for method in METHODS}   # filled below, consumed by Chapter 6.5


def wls_diag(base_crop, mon_crop, dz_g, dx_g, kz_cent, kz_fac, kx_fac, wls_thr,
             pad_fac=PAD_FAC, wls_pow=1):
    """Zero-padded WLS cross-spectrum phase-plane fit; returns the fitted shift plus
    everything needed to draw the 5-panel diagnostic below."""
    Nz, Nx = base_crop.shape
    Nz_pad, Nx_pad = Nz * pad_fac, Nx * pad_fac
    kz_ax = np.fft.fftfreq(Nz_pad, d=dz_g) * 2 * np.pi
    kx_ax = np.fft.fftfreq(Nx_pad, d=dx_g) * 2 * np.pi
    KZ, KX = np.meshgrid(kz_ax, kx_ax, indexing='ij')
    taper = np.outer(tukey(Nz, alpha=0.15), tukey(Nx, alpha=0.15))
    XS = (np.fft.fft2(base_crop * taper, s=(Nz_pad, Nx_pad)) *
          np.conj(np.fft.fft2(mon_crop * taper, s=(Nz_pad, Nx_pad))))
    w = np.abs(XS)
    phi = np.angle(XS)
    band = (np.abs(KZ) < kz_fac * kz_cent) & (np.abs(KX) < kx_fac * kz_cent)
    mask = (w > wls_thr * w.max()) & band & ((np.abs(KZ) + np.abs(KX)) > 0)
    n_mask = int(mask.sum())
    if n_mask < 3:
        dz_est = dx_est = phi_0 = 0.0
    else:
        W = w[mask] ** wls_pow
        A = np.column_stack([KZ[mask], KX[mask], np.ones(n_mask)])
        c = np.linalg.lstsq(A * W[:, None], phi[mask] * W, rcond=None)[0]
        dz_est, dx_est, phi_0 = float(c[0]), float(c[1]), float(c[2])

    fitted = KX * dx_est + KZ * dz_est + phi_0
    phi_shift, w_shift = np.fft.fftshift(phi), np.fft.fftshift(w)
    kz_disp, kx_disp = np.fft.fftshift(kz_ax), np.fft.fftshift(kx_ax)
    band_shift = np.fft.fftshift(band)
    fitted_shift = np.fft.fftshift(fitted)
    phi_masked = np.where(band_shift & (np.fft.fftshift(w) > wls_thr * w.max()), phi_shift, np.nan)
    w_plot = np.where(band_shift, w_shift, np.nan)
    fitted_masked = np.where(band_shift, fitted_shift, np.nan)

    phi_kz_resid = phi - KX * dx_est
    phi_1d = np.zeros(Nz_pad); w_1d = np.zeros(Nz_pad)
    band_kx = np.abs(kx_ax) < kx_fac * kz_cent
    for i in range(Nz_pad):
        sel = band_kx & (w[i, :] > wls_thr * w.max())
        if sel.sum() > 0:
            phi_1d[i] = np.average(phi_kz_resid[i, :][sel], weights=w[i, :][sel])
            w_1d[i] = w[i, :][sel].sum()
    kz_1d_s, phi_1d_s, w_1d_s = np.fft.fftshift(kz_ax), np.fft.fftshift(phi_1d), np.fft.fftshift(w_1d)

    phi_kx_resid = phi - KZ * dz_est
    phi_1d_kx = np.zeros(Nx_pad); w_1d_kx = np.zeros(Nx_pad)
    band_kz_fit = np.abs(kz_ax) < kz_fac * kz_cent
    for j in range(Nx_pad):
        sel = band_kz_fit & (w[:, j] > wls_thr * w.max())
        if sel.sum() > 0:
            phi_1d_kx[j] = np.average(phi_kx_resid[:, j][sel], weights=w[:, j][sel])
            w_1d_kx[j] = w[:, j][sel].sum()
    kx_1d_s, phi_1d_kx_s, w_1d_kx_s = np.fft.fftshift(kx_ax), np.fft.fftshift(phi_1d_kx), np.fft.fftshift(w_1d_kx)

    return dict(
        dz_est=dz_est, dx_est=dx_est, phi_0=phi_0, n_mask=n_mask,
        kz_disp=kz_disp, kx_disp=kx_disp, phi_masked=phi_masked, w_plot=w_plot,
        w_max=w.max(), wls_thr=wls_thr, fitted_masked=fitted_masked,
        kz_1d_s=kz_1d_s, phi_1d_s=phi_1d_s, w_1d_s=w_1d_s,
        kx_1d_s=kx_1d_s, phi_1d_kx_s=phi_1d_kx_s, w_1d_kx_s=w_1d_kx_s,
        kz_fac=kz_fac, kx_fac=kx_fac, kz_cent=kz_cent,
    )


def plot_diag_row(axes_row, diag, stage_name, run_a, run_b):
    kz_disp, kx_disp = diag['kz_disp'], diag['kx_disp']
    kz_fac, kx_fac, kz_cent = diag['kz_fac'], diag['kx_fac'], diag['kz_cent']

    ax = axes_row[0]
    im = ax.imshow(diag['phi_masked'], aspect='auto', cmap='RdBu_r',
                    extent=[kx_disp[0], kx_disp[-1], kz_disp[-1], kz_disp[0]],
                    vmin=-np.pi, vmax=np.pi, origin='upper')
    plt.colorbar(im, ax=ax, label='phase [rad]', fraction=0.046, pad=0.04)
    ax.set_title(f'Cross-spectrum phase ({diag["n_mask"]} px)')
    ax.set_xlim(kx_disp[0] / 3, kx_disp[-1] / 3); ax.set_ylim(kz_disp[-1] / 3, kz_disp[0] / 3)

    ax = axes_row[1]
    im = ax.imshow(diag['w_plot'], aspect='auto', cmap='inferno',
                    extent=[kx_disp[0], kx_disp[-1], kz_disp[-1], kz_disp[0]], origin='upper')
    plt.colorbar(im, ax=ax, label='|XS| [a.u.]', fraction=0.046, pad=0.04)
    ax.contour(kx_disp, kz_disp, np.nan_to_num(diag['w_plot']),
               levels=[diag['wls_thr'] * diag['w_max']], colors='cyan', linewidths=0.8)
    ax.set_title(f'Energy (thr={diag["wls_thr"]:.2f}xmax)')
    ax.set_xlim(kx_disp[0] / 3, kx_disp[-1] / 3); ax.set_ylim(kz_disp[-1] / 3, kz_disp[0] / 3)

    ax = axes_row[2]
    im = ax.imshow(diag['fitted_masked'], aspect='auto', cmap='RdBu_r',
                    extent=[kx_disp[0], kx_disp[-1], kz_disp[-1], kz_disp[0]],
                    vmin=-np.pi, vmax=np.pi, origin='upper')
    plt.colorbar(im, ax=ax, label='phase [rad]', fraction=0.046, pad=0.04)
    ax.set_title(f'Fitted plane  dz={diag["dz_est"]:+.3f} m  dx={diag["dx_est"]:+.3f} m')
    ax.set_xlim(kx_disp[0] / 3, kx_disp[-1] / 3); ax.set_ylim(kz_disp[-1] / 3, kz_disp[0] / 3)

    ax = axes_row[3]
    kz_1d_s, phi_1d_s, w_1d_s = diag['kz_1d_s'], diag['phi_1d_s'], diag['w_1d_s']
    in_band = np.abs(kz_1d_s) < kz_fac * kz_cent
    sel = in_band & (w_1d_s > 0)
    sc = ax.scatter(kz_1d_s[sel], phi_1d_s[sel], c=w_1d_s[sel], cmap='viridis', s=14)
    plt.colorbar(sc, ax=ax, label='weight', fraction=0.046, pad=0.04)
    kz_fit = kz_1d_s[in_band]
    ax.plot(kz_fit, kz_fit * diag['dz_est'] + diag['phi_0'], 'r-', lw=1.3)
    ax.axhline(0, color='k', lw=0.5, ls='--')
    ax.set_title('1-D $k_z$ slice: measured vs fitted')
    ax.set_xlabel('$k_z$ [rad/m]'); ax.set_ylabel('phase [rad]')

    ax = axes_row[4]
    kx_1d_s, phi_1d_kx_s, w_1d_kx_s = diag['kx_1d_s'], diag['phi_1d_kx_s'], diag['w_1d_kx_s']
    in_band_kx = np.abs(kx_1d_s) < kx_fac * kz_cent
    sel = in_band_kx & (w_1d_kx_s > 0)
    sc = ax.scatter(kx_1d_s[sel], phi_1d_kx_s[sel], c=w_1d_kx_s[sel], cmap='viridis', s=14)
    plt.colorbar(sc, ax=ax, label='weight', fraction=0.046, pad=0.04)
    kx_fit = kx_1d_s[in_band_kx]
    ax.plot(kx_fit, kx_fit * diag['dx_est'] + diag['phi_0'], 'r-', lw=1.3)
    ax.axhline(0, color='k', lw=0.5, ls='--')
    ax.set_title('1-D $k_x$ slice: measured vs fitted')
    ax.set_xlabel('$k_x$ [rad/m]'); ax.set_ylabel('phase [rad]')

    axes_row[0].set_ylabel(f'{stage_name}\n(prof {run_a}->{run_b})\n$k_z$ [rad/m]')


def plot_method_diagnostics(method):
    """Chapter 6.4 driver: one figure per migration method -- 4 rows (stages) x 5
    columns (diagnostic panels). Stores each stage's (dz_est, dx_est) into
    PHASE_RESULTS[method] for Chapter 6.5's combined summary."""
    is_bp = (method == 'Back-propagation')
    kx_fac  = KX_FAC_BP  if is_bp else KX_FAC_IMG
    wls_thr = WLS_THR_BP if is_bp else WLS_THR_IMG
    wls_pow = WLS_POW_BP if is_bp else WLS_POW_IMG

    fig, axes = plt.subplots(4, 5, figsize=(24, 16))
    for row, (stage_name, run_a, run_b) in enumerate(STAGE_PAIRS):
        crop = get_crop(method, run_a, run_b)
        if crop is None:
            for ax in axes[row]:
                ax.text(0.5, 0.5, 'missing data', ha='center', va='center', transform=ax.transAxes)
            continue
        kz_fac = 0.5 if is_bp else KZ_FAC_OVERRIDE.get(stage_name, 0.5)
        diag = wls_diag(crop['base'], crop['mon'], crop['dz_g'], crop['dx_g'], crop['kz_cent'],
                         kz_fac, kx_fac, wls_thr, wls_pow=wls_pow)
        plot_diag_row(axes[row], diag, stage_name, run_a, run_b)
        PHASE_RESULTS[method][stage_name] = dict(dz=diag['dz_est'], dx=diag['dx_est'], n_mask=diag['n_mask'])
        print(f'  [{method}] {stage_name} ({run_a}->{run_b}): '
              f'dz={diag["dz_est"]:+.4f} m  dx={diag["dx_est"]:+.4f} m  n_mask={diag["n_mask"]}')

    fig.suptitle(f'{method}: WLS cross-spectrum phase-plane diagnostics per stage '
                 f'(ROI depth {ROI_PHYS[0]:.0f}-{ROI_PHYS[1]:.0f} m, radial {ROI_PHYS[2]}-{ROI_PHYS[3]} m)', y=1.01)
    plt.tight_layout(); plt.show(); plt.close(fig)


for method in METHODS:
    print(f'--- {method} ---')
    plot_method_diagnostics(method)


_____
# Chapter 6.5 Summary of the Results

Compiling the four-stage displacement estimates (Chapter 6.4) from all three
migration methods into one comparison: a results table and a combined trajectory plot
with Kirchhoff-BP, Gazdag and Back-propagation overlaid on the same axes.

In [ ]:
summary_rows = []
for method in METHODS:
    for stage_name, run_a, run_b in STAGE_PAIRS:
        r = PHASE_RESULTS[method].get(stage_name)
        if r is None:
            continue
        summary_rows.append(dict(stage=stage_name, pair=f'{run_a}->{run_b}', method=method,
                                  dz_m=r['dz'], dx_m=r['dx'], n_mask=r['n_mask']))
summary_df = pd.DataFrame(summary_rows)

pivot_dz = summary_df.pivot(index='stage', columns='method', values='dz_m').loc[[s[0] for s in STAGE_PAIRS]]
pivot_dx = summary_df.pivot(index='stage', columns='method', values='dx_m').loc[[s[0] for s in STAGE_PAIRS]]
pivot = pd.concat({'dz (m)': pivot_dz, 'dx (m)': pivot_dx}, axis=1)


def style_error_table(df, value_cols, decimals=3):
    """Diverging red/blue table styling (white ~ 0, saturating at the largest value in
    the table) -- same convention as Hypothesis_2.ipynb's style_error_table."""
    vmax = np.nanmax(np.abs(df[value_cols].to_numpy(dtype=float)))
    vmax = vmax if vmax > 0 else 1.0
    fmt = {c: (lambda v, d=decimals: '-' if pd.isna(v) else f'{v:+.{d}f}') for c in value_cols}
    return df.style.format(fmt).background_gradient(cmap='RdBu_r', subset=value_cols, vmin=-vmax, vmax=vmax)


display(style_error_table(pivot, pivot.columns.tolist()))

# ── Combined movement summary plot: all three methods overlaid ─────────────────────
method_colours = {'Kirchhoff-BP': 'steelblue', 'Gazdag': 'darkorange', 'Back-propagation': 'seagreen'}
method_markers = {'Kirchhoff-BP': 'o', 'Gazdag': '^', 'Back-propagation': 's'}
stage_names = [s[0] for s in STAGE_PAIRS]
pair_labels = [f'{s[1]}->{s[2]}' for s in STAGE_PAIRS]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
for ax in (ax1, ax2):
    for i, stage_name in enumerate(stage_names):
        ax.axvspan(i - 0.5, i + 0.5, color=STAGE_COLOURS[stage_name], alpha=0.3, zorder=0, lw=0)
    ax.axhline(0, color='k', lw=0.6, ls='--')

for method in METHODS:
    dz_vals = [PHASE_RESULTS[method].get(s, {}).get('dz', np.nan) for s in stage_names]
    dx_vals = [PHASE_RESULTS[method].get(s, {}).get('dx', np.nan) for s in stage_names]
    ax1.plot(dz_vals, marker=method_markers[method], color=method_colours[method], lw=1.6,
             label=method, markersize=7)
    ax2.plot(dx_vals, marker=method_markers[method], color=method_colours[method], lw=1.6,
             label=method, markersize=7)

stage_handles = [Patch(facecolor=STAGE_COLOURS[s], alpha=0.6, label=s, edgecolor='grey', lw=0.5)
                 for s in stage_names]
ax1.legend(handles=stage_handles, loc='upper right', fontsize=8, framealpha=0.7, title='Stage')
ax2.legend(loc='lower right', fontsize=8, framealpha=0.7, title='Method')
ax1.set_ylabel('$\\Delta z$ (depth) [m]')
ax1.set_title('WLS phase-plane displacement estimate per stage -- all three migration methods')
ax2.set_ylabel('$\\Delta x$ (radial) [m]')
ax2.set_xlabel('Stage (profile pair)')
ax2.set_xticks(range(len(pair_labels)))
ax2.set_xticklabels([f'{s}\n({p})' for s, p in zip(stage_names, pair_labels)])
plt.tight_layout(); plt.show(); plt.close(fig)

kb_gz_dz_agree = np.corrcoef(pivot_dz['Kirchhoff-BP'], pivot_dz['Gazdag'])[0, 1]
print(
    f"Summary: Kirchhoff-BP and Gazdag agree closely on both the sign and the relative "
    f"size of the depth displacement in every stage (dz correlation = {kb_gz_dz_agree:+.2f}), "
    f"largest during Push ({pivot_dz.loc['Pushing', 'Kirchhoff-BP']:+.2f} m / "
    f"{pivot_dz.loc['Pushing', 'Gazdag']:+.2f} m) and near-zero during Wait "
    f"({pivot_dz.loc['Waiting', 'Kirchhoff-BP']:+.2f} m / {pivot_dz.loc['Waiting', 'Gazdag']:+.2f} m) -- "
    "expected, since both operate on the same underlying B-scan. Back-propagation's estimates "
    f"are visibly noisier (more scattered 1-D slices in Chapter 6.4) and disagree with "
    f"Kirchhoff-BP/Gazdag on the sign of dz during Push and Chase "
    f"({pivot_dz.loc['Pushing', 'Back-propagation']:+.2f} m vs. positive for both other methods), "
    "consistent with the lower SNR of the back-propagation reconstruction (Chapter 6.2's "
    "injection-halo artefact) and with Chapter 5's finding that back-propagation's phase-plane "
    "estimate is more sensitive to the specifics of the migrated/reconstructed image than "
    "Kirchhoff or Gazdag. All three methods qualitatively agree that the dominant displacement "
    "is a downward, borehole-ward-converging motion during Push that decays through Chase, "
    "Wait and Pull -- the fluid-injection kinematics the experiment was designed to produce."
)
